In [ ]:
import os 
import sys
import re 
import cv2  
import tqdm 
import logging
import json
import glob
import tqdm
import pickle as pkl  

wd = r'C:\Users\ADMIN\Documents\SAM\Research'
os.chdir(wd)

import torch 
import numpy as np 
import torch.nn as nn 
import torchvision 
import fiftyone as fo
import fiftyone.zoo as foz 
import fiftyone.types as fot
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image
from typing import List, Any 
from utils.dataLoader import COCOLoader, VOCPascalLoader, ADE20KLoader 
from utils.modelLoader import SAM1 
from utils.metrics import Metrics 

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
    )



## **CONFIGURATIONS**

In [2]:
# DataLoader configurations

# COCO Dataset 
coco_datapath = r'C:\\Users\\ADMIN\\fiftyone\\coco-2017\\validation\\data'
coco_labelpath = r'C:\Users\ADMIN\fiftyone\coco-2017\raw\instances_val2017.json' 
coco_dataset = COCOLoader(coco_datapath=coco_datapath, coco_labelpath=coco_labelpath)

# VOC Dataset 
voc_dataset = VOCPascalLoader(split='val') 

# ADE20K 
ade_dataset = ADE20KLoader(split='validation')


 100% |███████████████| 5000/5000 [1.1m elapsed, 0s remaining, 84.7 samples/s]       


2026-03-24 19:49:44,467 - INFO -  100% |███████████████| 5000/5000 [1.1m elapsed, 0s remaining, 84.7 samples/s]       
2026-03-24 19:49:44,547 - INFO - COCOLoader | images found: 4952
2026-03-24 19:49:44,558 - INFO - VOCPascal Loader | split: val | images found: 1449
2026-03-24 19:49:44,559 - WARNING - index_ade20k.pkl not found — class labels will fall back to numeric ids.
2026-03-24 19:49:44,571 - INFO - ADE20KLoader | split=validation | images found=2000


In [3]:
# Model Initialize 
model_type = 'vit_b'
checkpoint = r'C:\Users\ADMIN\Documents\SAM\Research\model\sam_vit_b_01ec64.pth'
sam_model = SAM1(model_name='SAM_1', model_type=model_type, checkpoint=checkpoint) 
sam_model

## **COCO Evaluation**

In [4]:
sample = coco_dataset[0]
sample

{'image_id': '69c288266cab34e0a9e2f683',
 'image': array([[[170, 136,  73],
         [173, 142,  77],
         [175, 144,  79],
         ...,
         [ 69,  76,  42],
         [ 68,  76,  39],
         [ 70,  71,  37]],
 
        [[172, 141,  77],
         [176, 145,  80],
         [177, 146,  81],
         ...,
         [ 69,  77,  40],
         [ 72,  80,  43],
         [ 71,  75,  40]],
 
        [[175, 144,  79],
         [177, 146,  81],
         [178, 147,  80],
         ...,
         [ 70,  78,  39],
         [ 69,  77,  40],
         [ 71,  75,  40]],
 
        ...,
 
        [[188, 189, 157],
         [183, 183, 149],
         [193, 187, 153],
         ...,
         [186, 157, 153],
         [186, 157, 153],
         [187, 156, 154]],
 
        [[186, 183, 152],
         [187, 184, 153],
         [186, 183, 152],
         ...,
         [198, 134, 134],
         [195, 120, 124],
         [186,  88, 101]],
 
        [[186, 183, 150],
         [187, 184, 151],
         [186, 183

In [ ]:
total_IoU = [] 
total_Dice = [] 
metrics = Metrics()

for sample in tqdm.tqdm(coco_dataset):
    sample_image = sample['image'] 
    sample_canvas_height = sample['canvas_height'] 
    sample_canvas_width = sample['canvas_width']
    sample_bounding_boxes = sample['bounding_boxes']
    sample_masks = sample['masks'] 
    sam_model._set_image(image=sample_image)

    sample_IoU = []
    sample_Dice = [] 
    for bbox, gt_masks in zip(sample_bounding_boxes, sample_masks): 
        input_bbox = np.array(bbox) 
        masks, scores, logits = sam_model._predict(input_bbox)
        # gt_overlay = np.zeros(shape=(sample_canvas_height, sample_canvas_width), dtype=np.int8)
        # gt_overlay[gt_masks] = 1
        iou = metrics._calculate_IoU(groundtruth_mask=gt_masks, predicted_mask=masks[0]) 
        dice = metrics._calculate_Dice(groundtruth_mask=gt_masks, predicted_mask=masks[0])
        sample_IoU.append(iou)
        sample_Dice.append(dice) 
        # print(f'IoU: {iou}') 
        # print(f'Dice: {dice}')
        # fig, ax = plt.subplots(1, 4, figsize=(20,5))
        # ax[0].imshow(sample_image)
        # ax[0].set_title("Original Image")

        # ax[1].imshow(masks[0])
        # ax[1].set_title("SAM Prediction")

        # ax[2].imshow(gt_masks)
        # ax[2].set_title("Ground Truth Mask")

        # ax[3].imshow(sample_image)
        # ax[3].imshow(masks[0], alpha=0.5)
        # ax[3].set_title("Prediction Overlay")


    mean_sample_iou = np.mean(sample_IoU) 
    mean_sample_dice = np.mean(sample_Dice)
    logging.info(f'mean sample IoU: {mean_sample_iou} | mean sample Dice: {mean_sample_dice}')  

    total_IoU.append(mean_sample_iou) 
    total_Dice.append(mean_sample_dice) 

coco_mean_IoU = np.mean(total_IoU)
coco_mean_Dice = np.mean(total_Dice)

logging.info(f'COCO mean IoU: {coco_mean_IoU} | COCO mean Dice: {coco_mean_Dice}')

In [ ]:
total_IoU = [] 
total_Dice = [] 
metrics = Metrics()

for sample in tqdm.tqdm(voc_dataset):
    sample_image = sample['image'] 
    sample_canvas_height = sample['canvas_height'] 
    sample_canvas_width = sample['canvas_width']
    sample_bounding_boxes = sample['bounding_boxes']
    sample_masks = sample['masks'] 
    sam_model._set_image(image=sample_image)

    sample_IoU = []
    sample_Dice = [] 
    for bbox, gt_masks in zip(sample_bounding_boxes, sample_masks): 
        input_bbox = np.array(bbox) 
        masks, scores, logits = sam_model._predict(input_bbox)
        # gt_overlay = np.zeros(shape=(sample_canvas_height, sample_canvas_width), dtype=np.int8)
        # gt_overlay[gt_masks] = 1
        iou = metrics._calculate_IoU(groundtruth_mask=gt_masks, predicted_mask=masks[0]) 
        dice = metrics._calculate_Dice(groundtruth_mask=gt_masks, predicted_mask=masks[0])
        sample_IoU.append(iou)
        sample_Dice.append(dice) 
        # print(f'IoU: {iou}') 
        # print(f'Dice: {dice}')
        # fig, ax = plt.subplots(1, 4, figsize=(20,5))
        # ax[0].imshow(sample_image)
        # ax[0].set_title("Original Image")

        # ax[1].imshow(masks[0])
        # ax[1].set_title("SAM Prediction")

        # ax[2].imshow(gt_masks)
        # ax[2].set_title("Ground Truth Mask")

        # ax[3].imshow(sample_image)
        # ax[3].imshow(masks[0], alpha=0.5)
        # ax[3].set_title("Prediction Overlay")


    mean_sample_iou = np.mean(sample_IoU) 
    mean_sample_dice = np.mean(sample_Dice)
    logging.info(f'mean sample IoU: {mean_sample_iou} | mean sample Dice: {mean_sample_dice}')  

    total_IoU.append(mean_sample_iou) 
    total_Dice.append(mean_sample_dice) 

coco_mean_IoU = np.mean(total_IoU)
coco_mean_Dice = np.mean(total_Dice)

logging.info(f'COCO mean IoU: {coco_mean_IoU} | COCO mean Dice: {coco_mean_Dice}')

## **VOC Pascal evaluation**

In [5]:
voc_sample = voc_dataset[0] 
voc_sample

{'image_id': '2007_000033',
 'image': array([[[221, 221, 221],
         [212, 212, 212],
         [215, 215, 215],
         ...,
         [226, 226, 226],
         [221, 221, 221],
         [223, 223, 223]],
 
        [[214, 214, 214],
         [205, 205, 205],
         [208, 208, 208],
         ...,
         [222, 222, 222],
         [217, 217, 217],
         [219, 219, 219]],
 
        [[217, 217, 217],
         [208, 208, 208],
         [210, 210, 210],
         ...,
         [223, 223, 223],
         [218, 218, 218],
         [220, 220, 220]],
 
        ...,
 
        [[ 98,  85,  69],
         [100,  87,  71],
         [ 94,  81,  65],
         ...,
         [198, 198, 198],
         [195, 193, 194],
         [195, 193, 194]],
 
        [[ 93,  80,  64],
         [ 92,  79,  63],
         [ 93,  80,  64],
         ...,
         [196, 196, 196],
         [194, 192, 193],
         [195, 193, 194]],
 
        [[ 86,  73,  57],
         [101,  88,  72],
         [103,  90,  74],
     

In [17]:
total_IoU = [] 
total_Dice = [] 
metrics = Metrics()

for sample in tqdm.tqdm(voc_dataset):
    sample_image = sample['image'] 
    sample_canvas_height = sample['canvas_height'] 
    sample_canvas_width = sample['canvas_width']
    sample_bounding_boxes = sample['bounding_boxes']
    sample_masks = sample['masks'] 
    sam_model._set_image(image=sample_image)

    sample_IoU = []
    sample_Dice = [] 
    for bbox, gt_masks in zip(sample_bounding_boxes, sample_masks): 
        input_bbox = np.array(bbox) 
        masks, scores, logits = sam_model._predict(input_bbox)
        # gt_overlay = np.zeros(shape=(sample_canvas_height, sample_canvas_width), dtype=np.int8)
        # gt_overlay[gt_masks] = 1
        iou = metrics._calculate_IoU(groundtruth_mask=np.int8(gt_masks), predicted_mask=masks[0]) 
        dice = metrics._calculate_Dice(groundtruth_mask=np.int8(gt_masks), predicted_mask=masks[0])
        sample_IoU.append(iou)
        sample_Dice.append(dice) 
        # print(f'IoU: {iou}') 
        # print(f'Dice: {dice}')
        # fig, ax = plt.subplots(1, 4, figsize=(20,5))
        # ax[0].imshow(sample_image)
        # ax[0].set_title("Original Image")

        # ax[1].imshow(masks[0])
        # ax[1].set_title("SAM Prediction")

        # ax[2].imshow(gt_masks)
        # ax[2].set_title("Ground Truth Mask")

        # ax[3].imshow(sample_image)
        # ax[3].imshow(masks[0], alpha=0.5)
        # ax[3].set_title("Prediction Overlay")


    mean_sample_iou = np.mean(sample_IoU) 
    mean_sample_dice = np.mean(sample_Dice)
    logging.info(f'mean sample IoU: {mean_sample_iou} | mean sample Dice: {mean_sample_dice}')  

    total_IoU.append(mean_sample_iou) 
    total_Dice.append(mean_sample_dice) 

coco_mean_IoU = np.mean(total_IoU)
coco_mean_Dice = np.mean(total_Dice)

logging.info(f'COCO mean IoU: {coco_mean_IoU} | COCO mean Dice: {coco_mean_Dice}')

  2%|▏         | 25/1449 [02:15<2:08:23,  5.41s/it]


KeyboardInterrupt: 

## **ADE20K Evaluation**

In [15]:
count = 0 
for sample in tqdm.tqdm(ade_dataset):
    count += 1

100%|██████████| 2000/2000 [01:04<00:00, 30.89it/s]
